[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/09_Manantiales_Bofedales/01_manantiales_bofedales.ipynb)

# Manantiales y bofedales: proxies topograficos e hidricos
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [1]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

Librerías listas.
earthengine-api: 1.7.39
geemap: 0.38.8


In [2]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "gen-lang-client-0636031685"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

Earth Engine inicializado correctamente.
Proyecto: gen-lang-client-0636031685


## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [3]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

Zona: Escoma, La Paz (prueba)
Ventana analítica: 1996-01-01 -> 2026-12-31 (31 años calendario)
Años completos para métricas: 1996 - 2025
Calendario: siembra mes 11 | cosecha mes 4
Umbrales activos: {'helada_c': 0.0, 'helada_grave_c': -2.0, 'mes_seco_mm': 30.0, 'spi_severo': -1.5, 'ndvi_anomalia_z': -1.5, 'manantial_occurrencia': 100, 'manantial_drenaje_max': 50, 'bofedal_altitud_min': 3000, 'bofedal_altitud_max': 5200, 'bofedal_pendiente_max': 5.0, 'bofedal_seasonality': 8, 'bofedal_ndvi_seco': 0.25, 'clases_agricolas': [12, 14]}


In [4]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

Utilidades: apilar(), reducir_bandas(), composite_mensual(), promediar_aoi(), serie_aoi()


In [5]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

Mounted at /content/drive
Descargando AOI desde Google Drive...


Downloading...
From: https://drive.google.com/uc?id=11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N
To: /content/zona_aoi.geojson
100%|██████████| 6.32k/6.32k [00:00<00:00, 3.30MB/s]


AOI cargado desde: /content/zona_aoi.geojson
Zona: Escoma, La Paz (prueba)
Área del AOI: 5,638.9 km²
Extensión (bbox): [-68.088142, -19.88671, -67.170426, -19.050138]
Puntos de saber local cargados: {'manantiales': 0, 'bofedales': 0}


Map(center=[-19.436538457151876, -67.61413826547425], controls=(WidgetControl(options=['position', 'transparen…

# M9. Manantiales: mapa de candidatos por criterios topográficos e hídricos (proxy)
**Dimensiones:** *ocurrencia* (agua permanente en cabecera), *proximidad* (drenaje bajo),
*criterio múltiple* (priorización).

> **⚠️ Advertencia:** no existe un dataset global de manantiales confiable y gratuito.
> Este módulo produce **candidatos** (hipótesis de campo) que **deben validarse** con las
> comunidades y, si es posible, con datos de caudal o sensor de humedad.

**Criterios (todos configurables en UMBRALES de CONFIG):**
1. **Agua permanente** en JRC: ocurrencia ≥ `manantial_occurrencia` (100 = siempre agua),
2. **Cabecera de cuenca:** acumulación de flujo ≤ `manantial_drenaje_max` celdas
   (poca área aguas arriba = nacimiento, no río grande),
3. **Pendiente moderada** (≤ 25°) para descartar acantilados sin acceso.

* **Requiere ejecutar previamente:** M1 (relieve y drenaje) y M8 (ocurrencia JRC).

<details>
<summary><b>📐 Cómo priorizar los candidatos en campo</b> — clic para ver</summary>
<br>

1. Ordenar la tabla por **ocurrencia** descendente (candidatos con agua más constante).
2. Descartar los que coinciden con embalses/presas o lagunas grandes ya inventariadas.
3. Contrastar con el **saber local** (`PUNTOS_LOCALES["manantiales"]` de CONFIG): los puntos
   conocidos deben caer cerca de candidatos — si no caen, los umbrales están demasiado estrictos.
4. Visitar con **caudalímetro o datalogger de humedad** los 10–20 primeros para confirmar.

</details>

In [6]:
# ============================================================
# M9.1 Criterios, polígonos candidatos y tabla priorizada
# ============================================================

occ_min = UMBRALES["manantial_occurrencia"]
dren_max = UMBRALES["manantial_drenaje_max"]

cand_mask = (
    ocurrencia.gte(occ_min)
    .And(drenaje.gt(0))              # excluir celdas sin dato (0)
    .And(drenaje.lte(dren_max))      # cabeceras: poco área aguas arriba
    .And(pend.lte(25))               # pendiente moderada
)

# Vectorización: agrupa píxeles contiguos en polígonos con nº de píxeles ("count")
candidatos = cand_mask.updateMask(cand_mask).reduceToVectors(
    reducer=ee.Reducer.count(),
    geometry=roi,
    scale=ESCALAS["jrc"],
    maxPixels=1e9,
    eightConnected=True,
)

# Enriquecer cada polígono: ocurrencia, altitud y pendiente medias
info_cand = (
    ee.Image.cat([
        ocurrencia.rename("occ"),
        elev.rename("alt"),
        pend.rename("pend"),
    ])
    .reduceRegions(candidatos, ee.Reducer.mean(), scale=ESCALAS["jrc"])
    .map(lambda f: f.set(
        "km2",
        ee.Number(f.get("count"))
        .multiply(ESCALAS["jrc"] ** 2)
        .divide(1e6),
    ))
)

n_cand = info_cand.size().getInfo()
print(f"Polígonos candidatos a manantiales: {n_cand}")


def _con_coords(f):
    c = f.geometry().centroid(maxError=30)
    return f.set({"lon": c.coordinates().get(0), "lat": c.coordinates().get(1)})


# Top 20: primero por ocurrencia (agua más constante)
top_cand = (
    ee.FeatureCollection(info_cand.map(_con_coords))
    .sort("occ", ascending=False)
    .limit(20)
)

fc_top = top_cand.getInfo()
df_mana = pd.DataFrame([
    {
        "lon": round(f["properties"].get("lon"), 5),
        "lat": round(f["properties"].get("lat"), 5),
        "occ_pct": round(f["properties"].get("occ") or 0, 1),
        "alt_msnm": round(f["properties"].get("alt") or 0, 0),
        "pend_grados": round(f["properties"].get("pend") or 0, 1),
        "km2": round(f["properties"].get("km2") or 0, 3),
    }
    for f in fc_top["features"]
])

display(df_mana)
print("Criterio de orden: ocurrencia descendente. Validar cada punto en campo/saber local.")

# --- Mapa ---------------------------------------------------------------------
Map_m9 = geemap.Map()
Map_m9.centerObject(roi, ZONA["zoom"])
Map_m9.add_basemap("HYBRID")
Map_m9.addLayer(elev, vis_elev, "Elevación (msnm)", False)
Map_m9.addLayer(
    info_cand.style({"color": "orange", "fillColor": "00000000", "width": 2}),
    {},
    f"Candidatos a manantiales ({n_cand})",
)
if PUNTOS_LOCALES["manantiales"]:
    Map_m9.addLayer(
        ee.Geometry.MultiPoint(PUNTOS_LOCALES["manantiales"]),
        {"color": "cyan"},
        "Manantiales conocidos (saber local)",
    )
Map_m9.addLayer(borde_aoi(), {}, "AOI")
Map_m9

NameError: name 'ocurrencia' is not defined

## Interpretación de M9
* **Los candidatos son hipótesis, no confirmaciones:** un manantial real necesita caudal
  sostenido o evidencia local; el satélite solo señala *dónde mirar primero*.
* **Cobertura de la red:** si los manantiales conocidos caen *dentro o muy cerca* de candidatos,
  los criterios funcionan para la zona; si caen lejos, relajar `manantial_occurrencia` (p. ej. 90)
  o subir `manantial_drenaje_max` en CONFIG y volver a ejecutar M9.1.
* **Priorización:** ocurrencia alta + área pequeña + altitud media = candidato de mejor uso
  (bombeo por gravedad, captación comunitaria).

**Preguntas guía**
1. ¿Todos los manantiales que usan aparecen en el mapa de candidatos o faltan algunos?
2. ¿Los candidatos en zonas muy húmedas (alta ocurrencia) corresponden a nacientes o a ríos?
3. ¿Dónde propondrían instalar sensores CLIMBAT para confirmar caudal? (sitios con mayor uso)

# M10. Bofedales: candidatos por planicie + agua + verde en estación seca
**Dimensiones:** *ocurrencia* (agua presente), *estacionalidad* (meses con agua),
*tendencia* (NDVI de estación seca año a año), *área* (km² por clase de humedad).

**Criterios de candidato (UMBRALES en CONFIG):**
1. **Altitud** entre `bofedal_altitud_min` y `bofedal_altitud_max` (planicie altoandina),
2. **Pendiente** ≤ `bofedal_pendiente_max` (planicie, no ladera),
3. **Agua** con estacionalidad JRC ≥ `bofedal_seasonality` meses,
4. **Vegetación en estación seca:** NDVI ≥ `bofedal_ndvi_seco` (verde cuando todo lo demás se seca).

* **Primario:** combinación de JRC (M8) + NDVI MODIS en estación seca (M7) + relieve (M1).
* **Alternativa (comentar/probar):** `ESA/WorldCover/v200` clase 90 "Humedad herbácea" (10 m,
  solo 2020/2021) — ver M12; o clasificación supervisada Sentinel-1+2 (pesada, estilo FAO).

> Requiere ejecutar **M1, M7 y M8** antes que este módulo.

In [ ]:
# ============================================================
# M10.1 Criterios combinados, áreas por clase y mapa de candidatos
# ============================================================

alt_ok = elev.gte(UMBRALES["bofedal_altitud_min"]).And(
    elev.lte(UMBRALES["bofedal_altitud_max"])
)
pend_ok = pend.lte(UMBRALES["bofedal_pendiente_max"])
humedo = estacionalidad.gte(UMBRALES["bofedal_seasonality"])
verde = ndvi_seco_map.gte(UMBRALES["bofedal_ndvi_seco"])

base = alt_ok.And(pend_ok)                       # planicie altoandina
cand_bof = base.And(humedo).And(verde)           # estricto: planicie + agua + verde en seca

# Clases de humedad dentro de la planicie (para priorizar)
clases_bof = {
    "c_bof": (cand_bof, "Bofedal candidato (estricto)"),
    "c_perm": (base.And(estacionalidad.gte(10)), "Con agua casi todo el año (10-12 meses)"),
    "c_estac": (
        base.And(estacionalidad.gte(UMBRALES["bofedal_seasonality"]))
        .And(estacionalidad.lt(10)),
        "Húmedo estacional (8-9 meses)",
    ),
    "c_verde_solo": (
        base.And(verde).And(estacionalidad.lt(UMBRALES["bofedal_seasonality"])),
        "Verde en seco sin agua JRC (riego/humedad freática)",
    ),
}

bof_imgs = [
    pix_km2.updateMask(masc).rename(clave)
    for clave, (masc, _) in clases_bof.items()
]
d_bof = reducir_bandas(
    bof_imgs, [clave for clave, _ in clases_bof.items()], ESCALAS["jrc"],
    reductor=ee.Reducer.sum(), params=REDUCIR_AREA,
)
df_bof = pd.DataFrame({
    "clase": [et for _, et in clases_bof.values()],
    "km2": [d_bof.get(cl, 0) or 0 for cl, _ in clases_bof.items()],
})
bof_km2 = float(df_bof.loc[0, "km2"])

print(f"Bofedales candidatos (estricto): {bof_km2:,.2f} km² "
      f"({100 * bof_km2 / area_km2:.2f}% del AOI)")
display(df_bof.round(2))

# --- Mapa ---------------------------------------------------------------------
vis_bof = {"min": 0, "max": 12,
           "palette": ["ffffff", "edf8fb", "ccece6", "99d8c9", "66c2a4",
                       "2ca25f", "006d2c"]}

Map_m10 = geemap.Map()
Map_m10.centerObject(roi, ZONA["zoom"])
Map_m10.add_basemap("HYBRID")
Map_m10.addLayer(ndvi_seco_map, {"min": 0, "max": 0.8,
                                 "palette": ["f7fcf5", "c7e9c0", "74c476", "238b45", "005a32"]},
                 "NDVI estación seca", False)
Map_m10.addLayer(estacionalidad, vis_bof, "Estacionalidad JRC (meses con agua)", False)
Map_m10.addLayer(cand_bof.selfMask(), {"palette": ["1b8a3c"]},
                 "Bofedales candidatos (estricto)")
if PUNTOS_LOCALES["bofedales"]:
    Map_m10.addLayer(
        ee.Geometry.MultiPoint(PUNTOS_LOCALES["bofedales"]),
        {"color": "magenta"},
        "Bofedales conocidos (saber local)",
    )
Map_m10.addLayer(borde_aoi(), {}, "AOI")
Map_m10

In [ ]:
# ============================================================
# M10.2 Estado anual del bofedal: NDVI de estación seca (tendencia)
# ============================================================

meses_sc = sorted(CALENDARIO["meses_estacion_seca"])
anios_bof = [y for y in anios_ndvi if y in ANIOS_COMPLETOS]

ndseco_imgs = [
    ndvi
    .filter(ee.Filter.calendarRange(y, y, "year"))
    .filter(ee.Filter.calendarRange(meses_sc[0], meses_sc[-1], "month"))
    .mean()
    .rename(f"n_{y}")
    for y in anios_bof
]
d_ndseco = reducir_bandas(
    ndseco_imgs, [f"n_{y}" for y in anios_bof], ESCALAS["modis"], tam=6,
)
df_ndvi_seco = pd.DataFrame({
    "anio": anios_bof,
    "ndvi_seco": [d_ndseco.get(f"n_{y}") for y in anios_bof],
}).dropna()

pend_ndvi_seco = np.polyfit(df_ndvi_seco["anio"], df_ndvi_seco["ndvi_seco"], 1)[0]
peor_bof = df_ndvi_seco.loc[df_ndvi_seco["ndvi_seco"].idxmin()]

print(f"Tendencia del NDVI en estación seca: {pend_ndvi_seco:+.4f} por año "
      f"({pend_ndvi_seco * 10:+.3f} por década)")
print(f"Año más débil del bofedal (NDVI seco mínimo): {int(peor_bof['anio'])} "
      f"(NDVI {peor_bof['ndvi_seco']:.2f})")

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.bar(df_ndvi_seco["anio"], df_ndvi_seco["ndvi_seco"],
       color=np.where(df_ndvi_seco["ndvi_seco"] < df_ndvi_seco["ndvi_seco"].mean(),
                      "#d32f2f", "#43a047"))
ax.axhline(df_ndvi_seco["ndvi_seco"].mean(), color="gray", ls="--", label="media")
ax.plot(df_ndvi_seco["anio"],
        np.polyval(np.polyfit(df_ndvi_seco["anio"], df_ndvi_seco["ndvi_seco"], 1),
                   df_ndvi_seco["anio"]),
        color="#ff9800", lw=2, label=f"tendencia {pend_ndvi_seco * 10:+.3f}/década")
ax.set_title("Condición del bofedal en estación seca: NDVI anual (menor = más estrés hídrico)")
ax.set_ylabel("NDVI estación seca")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

res_m10 = {
    "bof_km2": bof_km2,
    "bof_pct": 100 * bof_km2 / area_km2,
    "bof_ndvi_seco_tend_decada": pend_ndvi_seco * 10,
    "bof_peor_anio": int(peor_bof["anio"]),
}

## Interpretación de M10
* **Candidato estricto** es un filtro conservador: si da 0 km², relajar umbrales (p. ej.
  `bofedal_seasonality` 8→6 o `bofedal_ndvi_seco` 0,25→0,20) y reejecutar.
* **"Verde en seco sin agua JRC":** vegetación que sobrevive a la seca sin agua superficial
  detectable → señala **riego, humedad freática o manantiales** (enlaza con M9).
* **Serie NDVI de seca (M10.2):** años por debajo de la media = deterioro temporal del bofedal;
  tendencia negativa sostenida = alerta de sobreuso o cambio climático; contrastar con caudal
  medido y con el uso que reportan las comunidades.

**Preguntas guía**
1. ¿Sus bofedales conocidos caen dentro de los candidatos? ¿faltan o sobran?
2. ¿El año más débil de la serie coincide con sequías recordadas o con mayor presión de pastoreo?
3. ¿Qué partes "verdes en seco" no son bofedales sino riego agrícola (M11)?

## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
